# <span style="color:red">ch2_ LLM활용의 기본 개념(Ollama)</span>

# 1. LLM을 활용하여 답변 생성하기

## 1) Ollama 이용한 로컬 LLM 이용
성능은 GPT, Claude 같은 모델보다 떨어지나, 개념 설명을 위해 open source 모델 사용

### ▶ ollama.com 다운로드 -> 설치 -> 모델 pull
- ollama pull deepseek-r1:1.5b (window키 + R => Powershell)

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="deepseek-r1:1.5b")
result = llm.invoke("What is the capital of South Korea?")
result

### ollama.com 다운로드 -> 설치 -> 모델 pull
- ollama pull llama3.2:1b (window키+R => powershell)
- llama : 공식적으로 한글지원 안 됨 (llama3.1 405b한글지원 가능-> llama3.3 70b)
- exaone : 공식적으로 한글지원

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")
result = llm.invoke("What is the capital of South Korea?")
result

In [ ]:
llm.invoke("한국 수도는 어디에요?")

## 2) openai 활용
- pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")
# result = llm.invoke("What is the capital of South Korea?")
# result

In [ ]:
# 환경 변수 가져오기
from dotenv import load_dotenv
import os
load_dotenv()
# os.getenv('OPENAI_API_KEY')

In [ ]:
# 코랩에서 OPENAI_API_KEY 읽어오기(.env못씀)
# 보안키 추가후
# from google.colab import userdata
# userdata.get('secretName')

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano",
                 # openai_api_key=os.getenv('OPENAI_API_KEY'),
                 )
llm.invoke("Where is the capital of Korea? Answer me in Korean")

In [ ]:
# 모든 모델의 키가 OPENAI_API_KEY는 아님
# Claude -> Anthropic
# Azure, upstage, Bedrock : 에러 메세지 참조하여 환경변수 생성

In [ ]:
# from langchain_openai import AzureOpenAI
# llm = AzureOpenAI(model="gpt-4o-mini")
# 에러를 내면 OPENAI_API_VERSION 환경변수가 필요하는 메세지

In [ ]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-3-5-sonnet-20240620")
# llm.invoke("What is the capital of Korea?")
# 에러 메세지를 봐도 환경변수 이름을 알 수 X -> ChatAnthropic 검색후, 
# langchain docs에서 명시한 ANTHROPIC_API_KEY 이름의 환경변수 설정

# 2. 렝체인 스타일로 프롬프트 작성하기
- 프롬프트 : llm호출시 쓰는 질문

In [ ]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")
llm.invoke("Where is the capital of Korea?")
# 프롬프트 타입 : 스트링

## 1) 기본 프롬프트 템플릿 사용
- PromptTemplate을 사용하여 변수가 포함된 템플릿 작성

In [ ]:
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model="llama3.2:1b")
prompt_template = PromptTemplate(
    template = "What is the capital of {country}", #{}안의 값을 새로운 값으로 대입 가능
    input_variables = ["country"]
)
prompt = prompt_template.invoke({"country":"Korea"})
print(prompt)
llm.invoke(prompt)

## 2) 메세지 기반 프롬프트 작성
- BaseMessage리스트
- BaseMessage 상속 받은 클래스 : AIMessage, HumanMessage, SystemMessage, ToolMessage

In [ ]:
# BaseMessage list로 하면 렝체인화X, ChatPromptTemplateX
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
llm = ChatOllama(model="llama3.2:1b")
message_list = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="What is the capital of Japan?"),
    AIMessage(content="The capital of Italy is Tokyo."),
    HumanMessage(content="What is the capital of Italy?"),
    AIMessage(content="The capital of Italy is Rome."),
    HumanMessage(content="What is the capital of Russia?")
]
from langchain_core.prompts import ChatPromptTemplate
chatPromptTemplate = ChatPromptTemplate.from_messages(message_list)
prompt = prompt_template.invoke({"country":"Korea"})
print(prompt)

## 3) ChatPromptTemplate 사용
- BaseMessage 리스트 -> 튜플 리스트

In [ ]:
# 위의 BaseMessage를 수정
chatPromptTemplate = ChatPromptTemplate.from_messages([
    ("system", "You ara a helpful assistant!"),
    ("human", "What is the capital of {country}?"),
])
country = input("어느 나라 수도가 궁금하세요?")
prompt = chatPromptTemplate.invoke({"country":country})
print(prompt)
result = llm.invoke(prompt)
result.content

In [ ]:
chatPromptTemplate = ChatPromptTemplate.from_messages([
    ("system", "당신은 대한민국 전문 도우미야!"),
    ("human", "{country}의 수도가 어디예요?"),
])
country = input("어느 나라 수도가 궁금하세요?")
prompt = chatPromptTemplate.invoke({"country":country})
print("프롬프트 :",prompt)
result = llm.invoke(prompt)
result.content

# 3. 답변 형식을 컨트롤하기
- invoke 실행결과는 AIMessage() -> String이나 json, 객체 : outputParser 이용

## 1) 문자열 출력 파서 이용
- StrOutputParser를 사용하여 LLM출력(AIMessage)을 단순 문자열로 변환

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template = "What is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
# 프롬프트 템플릿에 값 주입
prompt = prompt_template.invoke({"country":"Korea"})
print(prompt)
result = llm.invoke(prompt)
print('llm 결과 :', type(result))
# 문자열 출력 파서를 이용하여 llm응답을 단순 문자열 변환
output_parser = StrOutputParser()
print('파서 결과 :', output_parser.invoke(result))

In [ ]:
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Korea"})))

In [ ]:
# PromptTemplate(변수설정) => ChatPromptTemplate(변수설정, system과 모법답안 지정)
llm = ChatOllama(model="llama3.2:1b")

chat_prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant with expertise in South Korea."),
    ("human", "What is the capital of {country}? Return the name if the city only.")
])

output_parser = StrOutputParser()

output_parser.invoke(llm.invoke(chat_prompt_template.invoke({"country":"Korea"})))

## 2) Json 출력 파서 이용
- json()으로 응답하기를 원하지만, 우선 어떤 형식으로 반환되는 확인
- {"name":"홍","age":22}(json) /{'name':'홍','age':22}(dict)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
country_detail_prompt = PromptTemplate(
    template="""Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dict only
    """,
    input_variables= ["country"]
)
prompt = country_detail_prompt.invoke({"country":"Korea"})
print(type(prompt), prompt)
# Json Output 파서
output_parser = JsonOutputParser()
ai_message = llm.invoke(prompt)
print(type(prompt), ai_message)
json_result = output_parser.invoke(ai_message)
print(type(json_result))

In [ ]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]    
)
output_parser = JsonOutputParser()
info = output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))
info


In [ ]:
type(info)

## 3) 구조화된 출력 사용
- Pydantic 모델을 사용하여 LLM 출력을 구조화된 형식으로 받기(JsonParser보다 훨씬 안정적)
- Pydantic : 데이터 유효성 검사, 설정관리를 간편하게 해주는 라이브러리

In [ ]:
from pydantic import BaseModel
class User:
    def __init__(self, id, name, is_active=True):
        self.id = id
        self.name = name
        self.is_active = is_active
user = User(1, "홍길동")
print(user)

In [ ]:
from pydantic import BaseModel, Field
class User(BaseModel):
    # gt:id>0, ge:id>=0, lt=0:id<0 , le=0:le<=0
    id:int   = Field(gt=0,              description="id")
    name:str = Field(min_length=2,      description="name")
    is_active:bool=Field(default=True,  description="id활성화")
user = User(id="1", name="홍길동")
print(user)

In [ ]:
country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}
    - Capital
    - Population
    - Language
    - Currency
    return it is JSON format and return the JSON dictionary only""",
    input_variables = ["country"]    
)
class CountryDetail(BaseModel): # description: 더 정확한 출력 유도
    capital:str    = Field(description="the capital of the country")
    population:int = Field(description="the population of the country")
    language:str   = Field(description="the language of the country")
    currency:str   = Field(description="the currency of the country")
# 출력 형식 파서 + LLM
structedllm = llm.with_structured_output(CountryDetail)

# output_parser = JsonOutputParser()
# output_parser.invoke(llm.invoke(country_detail_prompt.invoke({"country":"Korea"})))

info = structedllm.invoke(country_detail_prompt.invoke({"country":"Korea"}))
type(info)

In [ ]:
print(info)
print(info.capital, info.population)

In [ ]:
print('info를 json :', info.model_dump_json()) # json()
print('info를 dict :', info.model_dump()) # dict()

# 4. LCEL을 활용한 렝체인 생성하기
## 1) 문자열 출력 파서 사용
- invoke

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
llm = ChatOllama(model="llama3.2:1b", 
                 temperature=0) # 
# 명시적인 지시사항이 포함된 프롬프트
prompt_template = PromptTemplate(
    template = "What is the capital of {country}. Return the name of the city only",
    input_variables = ["country"]
)
output_parser = StrOutputParser()
output_parser.invoke(llm.invoke(prompt_template.invoke({"country":"Korea"})))

## 2) LCEL을 사용한 간단한 체인 구성
- 파이프연산자(|) 사용

In [ ]:
# 프롬프트템플릿 -> llm -> 출력파서를 연결하는 체인 생성
capital_chain = prompt_template | llm | output_parser
# 생성된 체인 invoke
capital_chain.invoke({"country":"Korea"})

In [ ]:
type(capital_chain)

## 3) 복합 체인 구성
- 여러단계의 추론이 필요한 경우 (체인 연결)

In [ ]:
# 나라 설명 -> 나라명
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
country_prompt = PromptTemplate(
    template="""Guess the name of the country based on the following information:
    {information}
    Return the name of the country only""",
    input_variables=["information"]
)
output_parser.invoke(llm.invoke(country_prompt.invoke({"information":
                                        "This country is very famous for its beer"})))

In [ ]:
# 나라명 추측 체인 생성
country_chain = country_prompt | llm | output_parser
# type(country_chain)
country_chain.invoke({"information":"This country is very famous for its beer"})

In [ ]:
# 나라설명을 입력 -> (나라명 ->) 그 나라 수도 출력
final_chain = country_chain | capital_chain
final_chain.invoke({"information":"This country is very famous for its beer"})

In [ ]:
final_chain = {"country" : country_chain} | capital_chain
final_chain.invoke({"information":"This country is very famous for its beer"})

In [ ]:
from langchain_core.runnables import RunnablePassthrough
final_chain = {"information":RunnablePassthrough()} | {"country":country_chain} | capital_chain
final_chain.invoke("This country is very famous for its beer")

In [ ]:
# 프롬프트 템플릿에 변수 2
# 나라 설명 -> 나라명
from langchain_core.prompts import PromptTemplate
country_prompt = PromptTemplate(
    template = """Guess the name of the country in the {continent} based 
    on the following information:
    {information} 
    Return the name of the the country only""",
    input_variables=["information","continent"]
)
# output_parser.invoke(llm.invoke(country_prompt.invoke({"information":
#                                      "This country is very famous for its wine",
#                                                       "continent":"Europe"})))
country_chain = country_prompt | llm | output_parser
country_chain.invoke({"information":"This country is very famous for its wine",
                      "continent":"Europe"})

In [ ]:
final_chain = {"country" : country_chain} | capital_chain
final_chain.invoke({"information":"This country is very famous for its wine", "continent":"Europe"})

In [ ]:
# Q. 나라명 -> (제일 유명한 음식 ->) 음식의 레시피